## 导入模块并初始化


In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

In [ ]:
# Tushare Token
TUSHARE_TOKEN = load_tushare_token()

# 数据库路径：直接放在项目目录下
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange="SSE",
)

print("数据库位置：", Path(DB_PATH).resolve())

## 2. 初始化基础数据
项目开始时先跑一次，用来获取：
场内基金基础信息
交易日历

In [ ]:
manager.initialize_basic_data(
    fund_market="E",      # 场内基金
    fund_status="L",      # 上市状态
    cal_start_date="20150101",
    cal_end_date=today_str(),
    exchange="SSE",
)

## 3. 查看当前本地资产列表

In [ ]:
instruments = manager.store.get_instruments(listed_only=True)
print(instruments.shape)
instruments.head()

## 4. 查看本地已收录的 ts_code 列表

In [ ]:
ts_codes = manager.store.get_ts_codes(listed_only=True)
len(ts_codes), ts_codes[:10]

5. 更新单个基金/ETF 日线数据

例如“515100.SH”

In [ ]:
df_515100 = manager.update_one_daily_price(
    ts_code="515100.SH",
    start_date="20180101",
    end_date=today_str(),
)
df_515100.tail()

## 6. 批量更新指定资产池

In [ ]:
watchlist = [
    "510300.SH",  # 沪深300ETF
    "511010.SH",  # 国债ETF
    "518880.SH",  # 黄金ETF
]

summary = manager.update_daily_prices(
    ts_codes=watchlist,
    start_date="20150101",
    end_date=today_str(),
)

summary

## 7. 对全库做增量更新

In [ ]:
summary_all = manager.update_daily_prices()
summary_all

## 8. 只刷新最近一段时间的数据
这个功能适合日常维护。
比如你已经有历史全量数据了，之后每天只想回补最近 30 天或 60 天。

In [ ]:
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    lookback_days=30,
    end_date=today_str(),
)
recent_summary

如果想对全体上市基金刷新最近 30 天：

In [ ]:
recent_summary_all = manager.refresh_recent_daily_prices(
    lookback_days=30,
    end_date=today_str(),
)
recent_summary_all

## 9. 检查本地价格数据覆盖范围

In [ ]:
coverage = manager.check_price_coverage(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"]
)
coverage

检查全库：

In [ ]:
coverage_all = manager.check_price_coverage()
coverage_all.head(20)

筛选出还没有数据的标的：

In [ ]:
coverage_all[coverage_all["has_data"] == 0].head(20)

## 10. 检查数据是否已经更新到最新交易日

In [ ]:
status = manager.check_latest_price_status(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    exchange="SSE",
)
status

筛选出未更新到最新交易日的资产：

In [ ]:
status[status["is_up_to_date"] == 0]

## 11. 读取本地已存储的日线数据
 读取单个标的

In [ ]:
prices_510300 = manager.store.get_daily_prices(
    ts_codes=["510300.SH"],
    start_date="20230101",
    end_date=today_str(),
)
prices_510300.tail()

读取多个标的

In [ ]:
prices = manager.store.get_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    start_date="20230101",
    end_date=today_str(),
)
prices.head()

## 12. 透视查看单个字段（比如 close）

In [ ]:
prices = manager.store.get_daily_prices(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    start_date="20230101",
    end_date=today_str(),
)

close_df = prices.pivot(index="trade_date", columns="ts_code", values="close").sort_index()
close_df.tail()

成交额矩阵

In [ ]:
amount_df = prices.pivot(index="trade_date", columns="ts_code", values="amount").sort_index()
amount_df.tail()

## 13. 查看交易日历

In [ ]:
trade_cal = manager.store.get_trade_calendar(
    exchange="SSE",
    start_date="20240101",
    end_date=today_str(),
)
trade_cal.tail()

只看开市日

In [ ]:
open_dates = manager.store.get_open_trade_dates(
    exchange="SSE",
    start_date="20240101",
    end_date=today_str(),
)
open_dates[-10:]

## 14. 查看某个标的的数据起止日期

In [ ]:
manager.store.get_price_date_range("510300.SH")

查看某个标的最新交易日：

In [ ]:
manager.store.get_latest_trade_date("510300.SH")

## 15. 查看哪些标的还没有本地日线数据

In [ ]:
missing_codes = manager.store.get_missing_ts_codes(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"]
)
missing_codes

查看全库

In [ ]:
missing_all = manager.store.get_missing_ts_codes()
len(missing_all)

## 16. 查看哪些标的数据为空或滞后
比如要所有标的至少更新到某天：

In [ ]:
stale_codes = manager.store.get_empty_or_stale_ts_codes(
    ts_codes=["510300.SH", "511010.SH", "518880.SH"],
    expected_latest_trade_date="20260320",
)
stale_codes

## 17.查看更新日志
查看最近的

In [ ]:
logs = manager.store.get_update_log(limit=20)
logs

只看日线更新日志：

In [ ]:
logs_daily = manager.store.get_update_log(
    table_name="daily_prices",
    limit=20,
)
logs_daily

只看某个标的

In [ ]:
logs_510300 = manager.store.get_update_log(
    table_name="daily_prices",
    ts_code="510300.SH",
    limit=20,
)
logs_510300

## 18. 最常用的日常维护流程示例

以后作为 notebook 的“每日更新入口”。

In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token
# Tushare Token
TUSHARE_TOKEN = load_tushare_token()


# 数据库路径：直接放在项目目录下
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20070101",
    default_exchange="SSE",
)
watchlist = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    "510500.SH",  # 中证500ETF (核心权益：中盘成长)
    "510170.SH",  # 50ETF (核心权益：大宗商品股票集合)
    "159915.SZ",  # 创业板ETF (核心权益：高新成长 - 纯被动)
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    "511010.SH",  # 国债ETF (基础配置：中长期债)
    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
    "159981.SZ",  # 能源ETF (抗通胀：能源/电力)
    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)
    "515100.SH",  # 红利低波100ETF (抗通胀：红利股)
]

# 1. 刷新最近 30 天数据
recent_summary = manager.refresh_recent_daily_prices(
    ts_codes=watchlist,
    lookback_days=30,
    end_date=today_str(),
)

print("最近数据刷新结果：")
print(recent_summary)

# 2. 检查最新状态
status = manager.check_latest_price_status(
    ts_codes=watchlist,
    exchange="SSE",
)

print("\n最新状态：")
display(status)

# 3. 读取最近一年价格
prices = manager.store.get_daily_prices(
    ts_codes=watchlist,
    start_date="20250101",
    end_date=today_str(),
)

print("\n最近价格数据：")
display(prices.tail(10))

## 19. 最常用的项目初始化流程示例

这个 cell 适合项目刚开始时跑一次。

In [ ]:
from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token
# Tushare Token
TUSHARE_TOKEN = load_tushare_token()


# 数据库路径：直接放在项目目录下
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20070101",
    default_exchange="SSE",
)

print("数据库位置：", Path(DB_PATH).resolve())
# Step 1: 初始化基础数据
manager.initialize_basic_data(
    fund_market="E",
    fund_status="L",
    cal_start_date="20070101",
    cal_end_date=today_str(),
    exchange="SSE",
)

# Step 2: 选定第一批资产池
watchlist = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    "510500.SH",  # 中证500ETF (核心权益：中盘成长)
    "510170.SH",  # 50ETF (核心权益：大宗商品股票集合)
    "159915.SZ",  # 创业板ETF (核心权益：高新成长 - 纯被动)
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    "511010.SH",  # 国债ETF (基础配置：中长期债)
    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
    "159981.SZ",  # 能源ETF (抗通胀：能源/电力)
    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)
    "515100.SH",  # 红利低波100ETF (抗通胀：红利股)
]


# Step 3: 拉取历史日线
summary = manager.update_daily_prices(
    ts_codes=watchlist,
    start_date="20070101",
    end_date=today_str(),
)

print("历史数据初始化完成：")
print(summary)

# Step 4: 检查覆盖范围
coverage = manager.check_price_coverage(ts_codes=watchlist)
display(coverage)

# Step 5: 检查是否更新到最新交易日
status = manager.check_latest_price_status(ts_codes=watchlist)
display(status)

## 20 新资产池智能补数（只补缺口，不重复拉取）
适用于：换了一套和旧资产池差异很大的新资产池，但其中部分标的、部分日期可能已经在本地数据库里。
这个 cell 会：
1. 自己初始化 manager（可独立运行）
2. 检查每个标的在指定区间内的本地覆盖情况
3. 按缺失交易日拆成连续区间
4. 只对缺口区间发起拉取，避免重复抓取已经存在的数据

In [ ]:

from pathlib import Path
import pandas as pd
from market_data import create_manager, today_str, load_tushare_token

# ========= 10年期回测资产池 (支持从 2014年8月 开始回测) =========
NEW_WATCHLIST = [
    "510300.SH",  # [保留] 沪深300ETF (华泰柏瑞) - 成立: 2012-05。核心权益。
    "513100.SH",  # [替换] 纳斯达克100ETF (国泰) - 成立: 2013-04。完美平替 159659，拥有极长历史的纳指标杆。
    "518880.SH",  # [保留] 黄金ETF (华安) - 成立: 2013-07。国内最早、流动性最好的黄金ETF。
    "511010.SH",  # [替换] 5年期国债ETF (国泰) - 成立: 2013-03。
                  # *注意: 30年期超长债ETF是近两年的新物种，历史回测只能用 5年期国债ETF 替代。
                  # 它的久期较短，暴跌时的向上凸性不如30年期，但相关性特征完全一致，是最好的避险基准。
    "510880.SH",  # [替换] 红利ETF (华泰柏瑞) - 成立: 2006-11。完美平替 515100，穿越牛熊的老牌防守利器。
    "162411.SZ",  # [替换] 华宝油气LOF - 成立: 2011-09。完美平替 501018，极具弹性的全球能源/通胀定价标的。
    "510170.SH",  # [替换] 大宗商品ETF (国联安) - 成立: 2010-12。
                  # *注意: 豆粕等纯商品期货ETF大多在2019年后才成立。这只ETF持有的是上证大宗商品类股票（煤炭、有色等），
                  # 虽然带有一定的股市Beta，但它是回测早期通胀周期的最佳长周期替代品。
]
START_DATE = "20130801"
END_DATE = today_str()
DB_PATH = "data/db/market_data.db"
EXCHANGE = "SSE"

# 可选：如果你怀疑交易日历没更新，可以保持 True
ENSURE_TRADE_CALENDAR = True

# ===== 独立初始化（无需依赖前置 cell） =====
TUSHARE_TOKEN = load_tushare_token()
manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date="20150101",
    default_exchange=EXCHANGE,
)

if ENSURE_TRADE_CALENDAR:
    _ = manager.update_trade_calendar(
        exchange=EXCHANGE,
        start_date=START_DATE,
        end_date=END_DATE,
    )

open_dates = manager.store.get_open_trade_dates(
    exchange=EXCHANGE,
    start_date=START_DATE,
    end_date=END_DATE,
)
if len(open_dates) == 0:
    raise ValueError("指定区间内没有拿到开市交易日，请先检查交易日历或日期参数。")

open_dates = [str(x) for x in open_dates]
open_pos = {d: i for i, d in enumerate(open_dates)}

# 本地基础信息（若数据库里已有 instruments，可用于排除上市前日期）
inst = manager.store.get_instruments(listed_only=False)
if len(inst) > 0 and "ts_code" in inst.columns:
    inst = inst.copy()
    if "list_date" in inst.columns:
        inst["list_date"] = inst["list_date"].astype(str)
    inst_map = inst.set_index("ts_code").to_dict("index")
else:
    inst_map = {}


def split_missing_ranges(missing_dates, pos_map):
    """把缺失交易日拆成连续区间，避免重复拉取。"""
    if not missing_dates:
        return []

    ordered = sorted(missing_dates, key=lambda x: pos_map[x])
    ranges = []
    start = ordered[0]
    prev = ordered[0]

    for d in ordered[1:]:
        if pos_map[d] == pos_map[prev] + 1:
            prev = d
        else:
            ranges.append((start, prev))
            start = d
            prev = d
    ranges.append((start, prev))
    return ranges


coverage_rows = []
fetch_plan_rows = []

for ts_code in NEW_WATCHLIST:
    meta = inst_map.get(ts_code, {})
    list_date = str(meta.get("list_date")) if meta.get("list_date") not in [None, "", "None", "nan"] else None

    effective_start = START_DATE
    if list_date is not None and list_date.isdigit():
        effective_start = max(START_DATE, list_date)

    expected_open_dates = [d for d in open_dates if d >= effective_start]
    expected_open_set = set(expected_open_dates)

    existing = manager.store.get_daily_prices(
        ts_codes=[ts_code],
        start_date=effective_start,
        end_date=END_DATE,
    )

    if len(existing) > 0:
        existing_dates = sorted(existing["trade_date"].astype(str).unique().tolist())
        existing_open_dates = [d for d in existing_dates if d in expected_open_set]
    else:
        existing_dates = []
        existing_open_dates = []

    missing_dates = [d for d in expected_open_dates if d not in set(existing_open_dates)]
    missing_ranges = split_missing_ranges(missing_dates, open_pos)

    coverage_rows.append({
        "ts_code": ts_code,
        "list_date": list_date,
        "effective_start": effective_start,
        "request_end": END_DATE,
        "has_any_local_data": int(len(existing_dates) > 0),
        "local_start": existing_dates[0] if existing_dates else None,
        "local_end": existing_dates[-1] if existing_dates else None,
        "expected_open_days": len(expected_open_dates),
        "local_open_days": len(existing_open_dates),
        "missing_open_days": len(missing_dates),
        "missing_ranges": "; ".join([f"{s}~{e}" for s, e in missing_ranges]) if missing_ranges else "",
    })

    for start_date, end_date in missing_ranges:
        fetch_plan_rows.append({
            "ts_code": ts_code,
            "fetch_start": start_date,
            "fetch_end": end_date,
        })

coverage_df = pd.DataFrame(coverage_rows).sort_values("ts_code").reset_index(drop=True)
fetch_plan_df = pd.DataFrame(fetch_plan_rows)

print("=== 本地覆盖检查 ===")
display(coverage_df)

if len(fetch_plan_df) == 0:
    print("当前新资产池在指定区间内已经完整覆盖，无需新增拉取。")
else:
    print("=== 待拉取缺口区间（只补缺口，不重复拉取） ===")
    display(fetch_plan_df)

    inserted_rows = 0
    updated_codes = set()
    fetch_logs = []

    for row in fetch_plan_df.itertuples(index=False):
        df_new = manager.update_one_daily_price(
            ts_code=row.ts_code,
            start_date=row.fetch_start,
            end_date=row.fetch_end,
        )
        n = len(df_new)
        inserted_rows += n
        if n > 0:
            updated_codes.add(row.ts_code)

        fetch_logs.append({
            "ts_code": row.ts_code,
            "fetch_start": row.fetch_start,
            "fetch_end": row.fetch_end,
            "inserted_rows": n,
        })

    fetch_log_df = pd.DataFrame(fetch_logs)

    print("=== 本次补数结果 ===")
    display(fetch_log_df)

    print({
        "requested_codes": len(NEW_WATCHLIST),
        "codes_with_gaps": int(fetch_plan_df["ts_code"].nunique()),
        "codes_updated": len(updated_codes),
        "inserted_rows": inserted_rows,
    })

    print("=== 补数后再次检查 ===")
    coverage_rows_after = []
    for ts_code in NEW_WATCHLIST:
        meta = inst_map.get(ts_code, {})
        list_date = str(meta.get("list_date")) if meta.get("list_date") not in [None, "", "None", "nan"] else None

        effective_start = START_DATE
        if list_date is not None and list_date.isdigit():
            effective_start = max(START_DATE, list_date)

        expected_open_dates = [d for d in open_dates if d >= effective_start]
        expected_open_set = set(expected_open_dates)

        existing = manager.store.get_daily_prices(
            ts_codes=[ts_code],
            start_date=effective_start,
            end_date=END_DATE,
        )

        if len(existing) > 0:
            existing_dates = sorted(existing["trade_date"].astype(str).unique().tolist())
            existing_open_dates = [d for d in existing_dates if d in expected_open_set]
        else:
            existing_dates = []
            existing_open_dates = []

        missing_dates = [d for d in expected_open_dates if d not in set(existing_open_dates)]

        coverage_rows_after.append({
            "ts_code": ts_code,
            "effective_start": effective_start,
            "request_end": END_DATE,
            "expected_open_days": len(expected_open_dates),
            "local_open_days": len(existing_open_dates),
            "missing_open_days": len(missing_dates),
        })

    coverage_after_df = pd.DataFrame(coverage_rows_after).sort_values("ts_code").reset_index(drop=True)
    display(coverage_after_df)
